# LAB 09 — Tokenizer & Label Alignment (`tokenizer_alignment.ipynb`)

Notebook này tập trung vào vấn đề cốt lõi khi chuyển từ **Sequence Classification (LAB 08)** sang **Token Classification (LAB 09)**: **Subword Tokenization & Label Alignment** (Mục 8–11, 16 và Experiment 1 ở Mục 17).

Mục tiêu:
1. Minh họa sự lệch pha giữa nhãn cấp từ (`word-level BIO labels`) và chuỗi token con (`subword tokens`) của Transformer.
2. Chọn, giải thích và áp dụng nhất quán một chiến lược căn chỉnh nhãn (**Alignment Strategy**), làm rõ vai trò của `IGNORE (-100)` trong hàm loss.
3. Triển khai pipeline alignment chuẩn với `is_split_into_words=True`, kiểm tra trực quan `tokens`, `input_ids`, `attention_mask`, `word_ids`, `labels`.
4. Chạy **Experiment 1 — Tokenization Analysis** trên tối thiểu 20 câu (ở đây dùng 60 câu CoNLL-2003 gồm cả câu ngắn và câu dài) để đo tỉ lệ tách subword $R$ và so sánh giữa câu ngắn/dài, entity/non-entity.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-cased"
IGNORE_INDEX = -100
LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
ID2LABEL = {i: lbl for i, lbl in enumerate(LABEL_LIST)}

try:
    dataset = load_dataset("lhoestq/conll2003")
except Exception:
    dataset = load_dataset("eriktks/conll2003", trust_remote_code=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {tokenizer.__class__.__name__} | Vocab size: {len(tokenizer)}")
print("Splits:", {k: len(v) for k, v in dataset.items()})

Tokenizer: DistilBertTokenizerFast | Vocab size: 28996
Splits: {'train': 14041, 'validation': 3250, 'test': 3453}


## 1. Vấn đề Subword Tokenization & Label Alignment (Mục 8–9)

Dataset CoNLL-2003 được gán nhãn theo **từng từ (word-level)**, còn Pretrained Transformer (như `distilbert-base-cased`, WordPiece tokenizer) lại cắt những từ ít gặp hoặc từ ghép thành nhiều **subword tokens** (ví dụ `playing` → `play`, `##ing`; hoặc tên riêng lạ như `Zwingmann` bị cắt nhỏ). Ngoài ra tokenizer còn chèn thêm các token đặc biệt như `[CLS]` ở đầu và `[SEP]` ở cuối câu.

Hệ quả: độ dài chuỗi `subword tokens` lớn hơn số lượng `word-level labels` ban đầu. Nếu không căn chỉnh (align), ta sẽ không biết gán nhãn gì cho các mảnh `##...` và `[CLS]`/`[SEP]`.

In [2]:
# Minh họa nhanh vấn đề tách subword (Mục 8)
demo_words = ["Washington", "playing", "Zwingmann", "SpaceX"]
for w in demo_words:
    subwords = tokenizer.tokenize(w)
    print(f"{w:<12} -> {subwords}")

# Thử trên 1 câu trong CoNLL-2003 train (sample 4 có 'Zwingmann')
sample = dataset["train"][4]
words = sample["tokens"]
word_labels = [LABEL_LIST[t] for t in sample["ner_tags"]]

encoded_single = tokenizer(words, is_split_into_words=True)
sub_tokens = tokenizer.convert_ids_to_tokens(encoded_single["input_ids"])

print(f"\nSố từ gốc (words): {len(words)}")
print(f"Số subword tokens (gồm [CLS], [SEP]): {len(sub_tokens)}")
print("Ví dụ một vài từ bị tách trong câu 4:")
word_ids = encoded_single.word_ids()
for idx, (tok, wid) in enumerate(zip(sub_tokens, word_ids)):
    if wid is not None and words[wid] in {"Zwingmann", "sheepmeat", "veterinary"}:
        print(f"  pos={idx:2d} | word_id={wid:2d} | word={words[wid]:<12} | subword={tok:<12} | word_label={word_labels[wid]}")

Washington   -> ['Washington']
playing      -> ['playing']
Zwingmann    -> ['Z', '##wing', '##mann']
SpaceX       -> ['Space', '##X']

Số từ gốc (words): 31
Số subword tokens (gồm [CLS], [SEP]): 39
Ví dụ một vài từ bị tách trong câu 4:
  pos=11 | word_id= 8 | word=veterinary   | subword=veterinary   | word_label=O
  pos=14 | word_id=11 | word=Zwingmann    | subword=Z            | word_label=I-PER
  pos=15 | word_id=11 | word=Zwingmann    | subword=##wing       | word_label=I-PER
  pos=16 | word_id=11 | word=Zwingmann    | subword=##mann       | word_label=I-PER
  pos=23 | word_id=18 | word=sheepmeat    | subword=sheep        | word_label=O
  pos=24 | word_id=18 | word=sheepmeat    | subword=##me         | word_label=O
  pos=25 | word_id=18 | word=sheepmeat    | subword=##at         | word_label=O


## 2. Lựa chọn Alignment Strategy & Giải thích `IGNORE` (Mục 10–11)

Trong các chiến lược alignment (Strategy A, B, C):
* **Chiến lược được chọn: Strategy A (First-subword labeling + `IGNORE = -100`)**.
  * Token đặc biệt (`word_id is None`, tức `[CLS]`, `[SEP]`, `[PAD]`) nhận nhãn `-100` (`IGNORE`).
  * Subword **đầu tiên** của mỗi từ nhận nhãn BIO gốc của từ đó (`B-X`, `I-X`, hoặc `O`).
  * Các subword **theo sau** của cùng một từ (`word_id == previous_word_id`) nhận nhãn `-100` (`IGNORE`).

* **Lý do chọn Strategy A:**
  1. Giữ đúng ranh giới 1–1 ở cấp từ: mỗi từ chỉ đóng góp đúng 1 nhãn vào loss, không làm phình số lượng nhãn `I-X` giả tạo ở những từ bị cắt thành 3–4 mảnh subword (so với Strategy B lan truyền `I-X`).
  2. Khi suy luận (inference), ta chỉ cần đọc dự đoán ở subword đầu tiên của từng `word_id` để khôi phục chính xác chuỗi nhãn BIO cho câu gốc — rất khớp với cách đánh giá entity-level của CoNLL-2003.
  3. Là chuẩn mặc định được dùng phổ biến trong Hugging Face Token Classification.

* **Ý nghĩa của `IGNORE (-100)` trong hàm loss (Mục 11):**
  $$L = -\sum_{i:\, y_i \neq \text{IGNORE}} \log P(y_i \mid x)$$
  `IGNORE` **không** có nghĩa là subword đó bị xóa bỏ! Các token như `##ing` hay `##mann` vẫn có `input_ids` và `attention_mask = 1`, vẫn đi qua toàn bộ các tầng self-attention để cung cấp thông tin ngữ nghĩa cho token đầu tiên. `IGNORE` chỉ có nghĩa là label tại vị trí đó được bỏ qua khi tính Cross-Entropy loss (PyTorch `CrossEntropyLoss(ignore_index=-100)`).

* **Phân tích ảnh hưởng của Strategy A:**
  * *Ưu điểm:* Không thiên vị loss về các từ dài/từ hiếm bị cắt nhiều mảnh; tỷ lệ nhãn thực thể không bị méo khi đánh giá token-level.
  * *Nhược điểm nhỏ:* Thông tin phân loại trực tiếp ở các mảnh đuôi không bị phạt trực tiếp qua loss (mặc dù gradient vẫn truyền ngược qua self-attention về các tầng dưới).

In [3]:
# Triển khai pipeline chuẩn theo Mục 16:
# tokens = tokenizer(words, is_split_into_words=True, truncation=True, padding=True)
# word_ids -> alignment -> token_labels

def tokenize_and_align_labels(examples, max_length=128):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding=True,
        max_length=max_length,
    )
    aligned_labels = []
    all_word_ids = []
    for i, label_seq in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        all_word_ids.append(word_ids)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(IGNORE_INDEX)
            elif word_idx != previous_word_idx:
                # Subword đầu tiên của từ -> giữ nhãn gốc
                label_ids.append(label_seq[word_idx])
            else:
                # Subword tiếp theo của cùng 1 từ -> IGNORE (-100)
                label_ids.append(IGNORE_INDEX)
            previous_word_idx = word_idx
        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    tokenized_inputs["word_ids"] = all_word_ids
    return tokenized_inputs

# Áp dụng trên một mini-batch gồm 2 câu để quan sát đầy đủ:
# tokens, input_ids, attention_mask, word_ids, labels
mini_batch = dataset["train"][:2]
aligned = tokenize_and_align_labels(mini_batch, max_length=32)

for b_idx in range(2):
    print(f"=== Câu {b_idx} ===")
    print("Original words :", mini_batch["tokens"][b_idx])
    print("Original labels:", [ID2LABEL[x] for x in mini_batch["ner_tags"][b_idx]])
    toks = tokenizer.convert_ids_to_tokens(aligned["input_ids"][b_idx])
    rows = []
    for tok, iid, mask, wid, lbl in zip(
        toks,
        aligned["input_ids"][b_idx],
        aligned["attention_mask"][b_idx],
        aligned["word_ids"][b_idx],
        aligned["labels"][b_idx],
    ):
        lbl_str = "IGNORE (-100)" if lbl == IGNORE_INDEX else f"{lbl} ({ID2LABEL[lbl]})"
        rows.append({
            "token": tok,
            "input_id": iid,
            "attention_mask": mask,
            "word_id": wid,
            "aligned_label": lbl_str,
        })
    df_view = pd.DataFrame(rows)
    display(df_view.head(20))

=== Câu 0 ===
Original words : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Original labels: ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']


,token,input_id,attention_mask,word_id,aligned_label
0,[CLS],101,1,NaN,IGNORE (-100)
1,EU,7270,1,0.0,3 (B-ORG)
2,rejects,22961,1,1.0,0 (O)
3,German,1528,1,2.0,7 (B-MISC)
4,call,1840,1,3.0,0 (O)
5,to,1106,1,4.0,0 (O)
6,boycott,21423,1,5.0,0 (O)
7,British,1418,1,6.0,7 (B-MISC)
8,la,2495,1,7.0,0 (O)
9,##mb,12913,1,7.0,IGNORE (-100)


=== Câu 1 ===
Original words : ['Peter', 'Blackburn']
Original labels: ['B-PER', 'I-PER']


,token,input_id,attention_mask,word_id,aligned_label
0,[CLS],101,1,NaN,IGNORE (-100)
1,Peter,1943,1,0.0,1 (B-PER)
2,Blackburn,14428,1,1.0,2 (I-PER)
3,[SEP],102,1,NaN,IGNORE (-100)
4,[PAD],0,0,NaN,IGNORE (-100)
5,[PAD],0,0,NaN,IGNORE (-100)
6,[PAD],0,0,NaN,IGNORE (-100)
7,[PAD],0,0,NaN,IGNORE (-100)
8,[PAD],0,0,NaN,IGNORE (-100)
9,[PAD],0,0,NaN,IGNORE (-100)


In [4]:
# Kiểm tra riêng 1 câu có nhiều subword splits (sample 4 trong train) để thấy rõ Strategy A
s4 = dataset["train"][4:5]
aligned4 = tokenize_and_align_labels(s4, max_length=64)
toks4 = tokenizer.convert_ids_to_tokens(aligned4["input_ids"][0])

rows4 = []
for tok, iid, mask, wid, lbl in zip(
    toks4,
    aligned4["input_ids"][0],
    aligned4["attention_mask"][0],
    aligned4["word_ids"][0],
    aligned4["labels"][0],
):
    orig_word = s4["tokens"][0][wid] if wid is not None else "<SPECIAL>"
    lbl_str = "IGNORE" if lbl == IGNORE_INDEX else ID2LABEL[lbl]
    rows4.append({
        "token": tok,
        "orig_word": orig_word,
        "input_id": iid,
        "attention_mask": mask,
        "word_id": wid,
        "label": lbl_str,
    })

df4 = pd.DataFrame(rows4)
print("Các dòng mà từ bị tách thành >= 2 subword (hoặc special tokens quanh vị trí bị tách):")
split_wids = {wid for wid, cnt in Counter(w for w in aligned4["word_ids"][0] if w is not None).items() if cnt > 1}
display(df4[df4["word_id"].isin(split_wids) | df4["word_id"].isna()].head(20))

Các dòng mà từ bị tách thành >= 2 subword (hoặc special tokens quanh vị trí bị tách):


,token,orig_word,input_id,attention_mask,word_id,label
0,[CLS],<SPECIAL>,101,1,NaN,IGNORE
2,','s,112,1,1.0,O
3,s,'s,188,1,1.0,IGNORE
9,','s,112,1,7.0,O
10,s,'s,188,1,7.0,IGNORE
14,Z,Zwingmann,163,1,11.0,I-PER
15,##wing,Zwingmann,7635,1,11.0,IGNORE
16,##mann,Zwingmann,4119,1,11.0,IGNORE
23,sheep,sheepmeat,8892,1,18.0,O
24,##me,sheepmeat,3263,1,18.0,IGNORE


## 3. Experiment 1 — Tokenization Analysis (Mục 17)

Yêu cầu: chọn tối thiểu 20 câu, thống kê `# words`, `# tokens`, `# subword splits`, tính tỉ số:
$$R = \frac{\#\text{subword tokens}}{\#\text{words}}$$
(trong đó `# subword tokens` là số token WordPiece sinh ra từ các từ gốc, không tính `[CLS]` và `[SEP]`; ta cũng ghi nhận thêm tổng số token gồm cả `[CLS]`/`[SEP]` để dễ đối chiếu).

Để phép so sánh giữa **câu ngắn vs câu dài** và **entity vs non-entity** được khách quan, ta lấy **60 câu** từ tập `test` của CoNLL-2003: gồm **30 câu ngắn** (`3 <= số từ <= 10`) và **30 câu dài** (`số từ >= 25`).

In [5]:
test_split = dataset["test"]

short_indices = [i for i, r in enumerate(test_split) if 3 <= len(r["tokens"]) <= 10][:30]
long_indices = [i for i, r in enumerate(test_split) if len(r["tokens"]) >= 25][:30]
selected_indices = short_indices + long_indices

records = []
word_records = []

for idx in selected_indices:
    row = test_split[idx]
    words = row["tokens"]
    tags = [LABEL_LIST[t] for t in row["ner_tags"]]
    group = "short" if len(words) <= 10 else "long"

    enc = tokenizer(words, is_split_into_words=True, truncation=False)
    word_ids = enc.word_ids()
    sub_tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])

    # Đếm số subword của từng word_id
    sub_counts = Counter(w for w in word_ids if w is not None)
    n_words = len(words)
    n_subword_tokens = sum(sub_counts.values())          # không tính [CLS], [SEP]
    n_total_tokens_with_special = len(enc["input_ids"])  # có [CLS], [SEP]
    n_split_words = sum(1 for c in sub_counts.values() if c > 1)
    n_extra_subwords = n_subword_tokens - n_words

    records.append({
        "test_idx": idx,
        "group": group,
        "n_words": n_words,
        "n_subword_tokens": n_subword_tokens,
        "n_tokens_with_special": n_total_tokens_with_special,
        "n_split_words": n_split_words,
        "n_extra_subwords": n_extra_subwords,
        "R": n_subword_tokens / n_words,
    })

    for wid, (w, tag) in enumerate(zip(words, tags)):
        n_subs = sub_counts[wid]
        is_ent = tag != "O"
        subs_list = [sub_tokens[pos] for pos, w_pos in enumerate(word_ids) if w_pos == wid]
        word_records.append({
            "word": w,
            "tag": tag,
            "is_entity": is_ent,
            "entity_type": tag.split("-")[-1] if is_ent else "O",
            "n_subwords": n_subs,
            "is_split": n_subs > 1,
            "subwords": " ".join(subs_list),
            "group": group,
        })

df_sent = pd.DataFrame(records)
df_words = pd.DataFrame(word_records)

print(f"Tổng số câu phân tích: {len(df_sent)} (ngắn: {len(short_indices)}, dài: {len(long_indices)})")
total_w = df_sent["n_words"].sum()
total_sub = df_sent["n_subword_tokens"].sum()
total_split_words = df_sent["n_split_words"].sum()
total_extra = df_sent["n_extra_subwords"].sum()
print(f"Tổng # words: {total_w}")
print(f"Tổng # subword tokens (không tính [CLS]/[SEP]): {total_sub}")
print(f"Tổng # từ bị tách (>1 subword): {total_split_words} ({total_split_words/total_w:.2%})")
print(f"Tổng # subword phát sinh thêm (splits): {total_extra}")
print(f"Tỉ lệ R chung = {total_sub / total_w:.4f}")

print("\n--- Bảng mẫu 10 câu đầu ---")
display(df_sent.head(10))

Tổng số câu phân tích: 60 (ngắn: 30, dài: 30)
Tổng # words: 1247
Tổng # subword tokens (không tính [CLS]/[SEP]): 1731
Tổng # từ bị tách (>1 subword): 266 (21.33%)
Tổng # subword phát sinh thêm (splits): 484
Tỉ lệ R chung = 1.3881

--- Bảng mẫu 10 câu đầu ---


,test_idx,group,n_words,n_subword_tokens,n_tokens_with_special,n_split_words,n_extra_subwords,R
0,2,short,6,13,15,2,7,2.166667
1,15,short,9,10,12,1,1,1.111111
2,24,short,4,7,9,2,3,1.750000
3,34,short,10,25,27,8,15,2.500000
4,35,short,6,13,15,2,7,2.166667
5,48,short,6,25,27,5,19,4.166667
6,49,short,4,11,13,2,7,2.750000
7,50,short,5,5,7,0,0,1.000000
8,51,short,7,10,12,1,3,1.428571
9,53,short,8,14,16,4,6,1.750000


In [6]:
# So sánh Câu ngắn vs Câu dài
summary_len = (
    df_sent.groupby("group")
    .agg(
        sentences=("test_idx", "count"),
        total_words=("n_words", "sum"),
        total_subword_tokens=("n_subword_tokens", "sum"),
        total_tokens_with_special=("n_tokens_with_special", "sum"),
        split_words=("n_split_words", "sum"),
        extra_subwords=("n_extra_subwords", "sum"),
        mean_R=("R", "mean"),
    )
)
summary_len["overall_R"] = summary_len["total_subword_tokens"] / summary_len["total_words"]
summary_len["split_word_pct"] = (summary_len["split_words"] / summary_len["total_words"] * 100).round(2)
print("=== So sánh Câu ngắn vs Câu dài ===")
display(summary_len)

# So sánh Entity vs Non-entity
summary_ent = (
    df_words.groupby("is_entity")
    .agg(
        total_words=("word", "count"),
        total_subword_tokens=("n_subwords", "sum"),
        split_words=("is_split", "sum"),
        mean_subwords_per_word=("n_subwords", "mean"),
        max_subwords=("n_subwords", "max"),
    )
)
summary_ent.index = ["Non-Entity (O)", "Entity (PER/ORG/LOC/MISC)"]
summary_ent["R"] = summary_ent["total_subword_tokens"] / summary_ent["total_words"]
summary_ent["split_word_pct"] = (summary_ent["split_words"] / summary_ent["total_words"] * 100).round(2)
print("\n=== So sánh Entity vs Non-Entity ===")
display(summary_ent)

# So sánh chi tiết theo từng loại Entity
summary_type = (
    df_words.groupby("entity_type")
    .agg(
        total_words=("word", "count"),
        total_subword_tokens=("n_subwords", "sum"),
        split_words=("is_split", "sum"),
    )
)
summary_type["R"] = summary_type["total_subword_tokens"] / summary_type["total_words"]
summary_type["split_word_pct"] = (summary_type["split_words"] / summary_type["total_words"] * 100).round(2)
print("\n=== Tỉ lệ tách subword theo từng loại Entity ===")
display(summary_type)

=== So sánh Câu ngắn vs Câu dài ===


,sentences,total_words,total_subword_tokens,total_tokens_with_special,split_words,extra_subwords,mean_R,overall_R,split_word_pct
group,,,,,,,,,
long,30,1041,1329,1389,167,288,1.224270,1.276657,16.04
short,30,206,402,462,99,196,1.962037,1.951456,48.06



=== So sánh Entity vs Non-Entity ===


,total_words,total_subword_tokens,split_words,mean_subwords_per_word,max_subwords,R,split_word_pct
Non-Entity (O),962,1148,104,1.193347,5,1.193347,10.81
Entity (PER/ORG/LOC/MISC),285,583,162,2.045614,7,2.045614,56.84



=== Tỉ lệ tách subword theo từng loại Entity ===


,total_words,total_subword_tokens,split_words,R,split_word_pct
entity_type,,,,,
LOC,57,89,12,1.561404,21.05
MISC,20,33,6,1.650000,30.00
O,962,1148,104,1.193347,10.81
ORG,18,30,6,1.666667,33.33
PER,190,431,138,2.268421,72.63


In [7]:
# Liệt kê những từ bị chia thành nhiều subword nhất trong 60 câu
top_splits = (
    df_words[df_words["n_subwords"] >= 2]
    .sort_values(by=["n_subwords", "is_entity"], ascending=[False, False])
    .drop_duplicates(subset=["word", "tag"])
    .head(20)
)
print("=== Top 20 từ bị chia nhiều subword nhất ===")
display(top_splits[["word", "tag", "n_subwords", "subwords"]])

=== Top 20 từ bị chia nhiều subword nhất ===


,word,tag,n_subwords,subwords
36,SKIING-WORLD,B-MISC,7,SK ##II ##NG - W ##OR ##LD
95,Collomb-Patton,I-PER,5,Col ##lo ##mb - Patton
185,Ljudmila,B-PER,5,L ##ju ##d ##mi ##la
275,Shkvyrin,I-PER,5,S ##h ##k ##vy ##rin
585,Cristofoletto,I-PER,5,C ##risto ##fo ##lett ##o
838,al-Boushi,I-PER,5,al - Bo ##ush ##i
859,Rihawiy,I-PER,5,R ##ih ##aw ##i ##y
5,1996-12-06,O,5,1996 - 12 - 06
35,FREESTYLE,O,5,F ##RE ##ES ##TY ##LE
38,MOGUL,O,5,M ##O ##G ##U ##L


### Nhận xét Experiment 1: Những loại từ nào thường bị chia thành nhiều subword?

Từ số liệu thực nghiệm trên 60 câu CoNLL-2003 ở trên, ta thấy rất rõ:
1. **Thực thể (Entity) bị tách nhiều hơn hẳn Non-entity (`O`):** Các từ `O` đa phần là từ tiếng Anh thông dụng (giới từ, động từ phổ thông, dấu câu) nên tỉ lệ tách thấp ($R = 1.193$, chỉ $10.81\%$ từ bị tách). Ngược lại, nhóm từ thuộc Entity có $R = 2.046$ với tới $56.84\%$ số từ bị cắt thành nhiều subword; riêng `PER` lên tới $R = 2.268$ ($72.63\%$ từ bị tách).
2. **Các nhóm từ hay bị chia nhiều subword nhất:**
   * **Tên người và địa danh không phải tiếng Anh gốc (foreign proper nouns):** Ví dụ `Ljudmila`, `Shkvyrin`, `Cristofoletto`, `al-Boushi`, `Rihawiy` đều bị WordPiece chặt thành 5 mảnh vì không nằm nguyên vẹn trong vocab pretrain tiếng Anh.
   * **Từ có gạch nối, dấu chấm, hoặc dạng mã/tiêu đề (`-`, `.`):** WordPiece tách dấu câu/gạch nối và chia thành nhiều mảnh như `SKIING-WORLD` (7 tokens), `Collomb-Patton` (5), `30-year-old` (5), `U.S.` (4), `AL-AIN` (4).
   * **Từ viết hoa toàn bộ (`ALL CAPS`):** Vì mô hình dùng bản **cased** (`distilbert-base-cased`), các tiêu đề viết hoa toàn bộ như `FREESTYLE`, `MOGUL`, `RESULTS`, `SYRIA`, `TIGNES` bị tách nhỏ hơn nhiều so với dạng viết thường tương ứng.
   * **Từ có hình thái phức tạp / từ ghép hiếm:** Ví dụ `misdirected` (5 tokens), `aggravating` (5 tokens), `sheepmeat` (3 tokens).
3. **Câu ngắn vs Câu dài:** Nhóm câu ngắn trong tập test lại có tỉ lệ tách cao hơn rõ rệt ($R = 1.951$, $48.06\%$ từ bị tách) so với câu dài ($R = 1.277$, $16.04\%$ từ bị tách). Lý do là nhiều câu ngắn đầu bản tin của CoNLL-2003 là dòng tiêu đề viết hoa toàn bộ hoặc tên bảng kết quả thể thao/địa danh (`SKIING-WORLD`, `AL-AIN`, `SYRIA`), nơi mật độ từ hiếm và ký tự viết hoa đặc biệt cao.